# 파일시스템과 중간 결과

FilesystemBackend로 조사 중간 결과를 실제 디스크에 저장하고 다시 읽습니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.

### 코드 읽기와 비교 경계

`FilesystemBackend(root_dir=...)`는 에이전트가 사용하는 `/research/source-a.md` 같은 가상 경로를 실습 폴더의 실제 경로로 대응시킵니다. 가상 경로의 `/`는 컴퓨터의 파일 시스템 루트를 뜻하지 않습니다. `write_file`은 문자열을 저장하고 성공 여부를 구조화된 결과로 반환하며, `read_file`은 필요할 때 내용을 다시 가져옵니다. 파일을 썼다는 사실만으로 다음 모델 호출이 자동으로 그 내용을 읽는 것은 아니므로, 읽기 호출과 전달 시점을 명시해야 합니다.

중간 결과를 파일에 두는 방법은 대화 상태에 모든 문자열을 유지하는 방법과 장단점이 다릅니다. 파일은 큰 자료를 이름과 경로로 인계하고 실행 뒤에도 검사하기 좋지만, 오래된 파일을 잘못 읽거나 여러 실행이 같은 경로를 덮어쓸 수 있습니다. 실행 ID별 폴더, 출처 ID, 생성 시각, 완료 상태를 함께 기록하면 어느 작업의 자료인지 구분할 수 있습니다. 비밀이나 개인정보가 있는 자료는 저장 경로의 접근 권한과 보존 기간도 정해야 합니다.


In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
cwd = Path.cwd().resolve()
LAB = next(
    parent / 'labs/part6/ch27-deepagents'
    for parent in [cwd, *cwd.parents]
    if (parent / 'labs/part6/ch27-deepagents/pyproject.toml').is_file()
)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

가상 경로를 실습 작업 폴더에 연결하고 중간 파일을 씁니다.

In [3]:
# 가상 경로를 실제 실습 작업 폴더에 연결하고 중간 파일을 씁니다.
from deepagents.backends import FilesystemBackend
workspace = LAB / 'workspace'
workspace.mkdir(parents=True, exist_ok=True)
backend = FilesystemBackend(root_dir=workspace)
result = backend.write('/research/source-a.md', '# 자료 A\n공식 문서에서 확인한 하네스 설명')
print(type(result).__name__, 'workspace/research/source-a.md')

WriteResult workspace/research/source-a.md


## 단계 2

저장한 파일의 본문과 가상 경로 목록을 다시 읽어 확인합니다.

In [4]:
# 저장한 중간 파일의 본문과 가상 경로 목록을 다시 읽습니다.
read = backend.read('/research/source-a.md')
print(read.file_data['content'])
print('목록:', [item['path'] for item in backend.ls('/research').entries])
assert read.file_data['content'].startswith('# 자료 A')
assert any(item['path'] == '/research/source-a.md' for item in backend.ls('/research').entries)

# 자료 A
공식 문서에서 확인한 하네스 설명
목록: ['/research/final.md', '/research/source-a.md']
